# RÉ-EXÉCUTION v1 — ResNet50 (Aigis)

> ⚠ **Ceci n'est PAS le notebook d'origine** (supprimé). C'est une **ré-exécution** du code `code/v1/` du dépôt, faite le jour de l'export (voir la date des sorties), avec les hyperparamètres d'origine sauvegardés (`models/.../Hyperparametres/`) et **sans nouvelle recherche d'hyperparamètres**. Les scores peuvent différer légèrement de ceux des fichiers `models/.../v1/Resultats/` (TensorFlow/CUDA non déterministes).

Avant d'exécuter : *Add Input* → `daikhiyassine/dataset` **avant de démarrer la session**, accélérateur GPU, Internet activé. Ensuite *Save Version → Save & Run All*, puis *File → Download notebook*.

In [1]:
!pip install -q keras-tuner

In [2]:
!nvidia-smi
!git clone https://github.com/Yacindaikhi/A-Cross-Architecture-Benchmark-of-DL-Models-for-Structural-Defect-Detection-on-Uncurated-Data.git repo
!git -C repo log -1 --format='%h %ad %s'

Sun Nov 22 22:58:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 1. Nettoyage + partition (MD5, fuites inter-splits, RGB, 256x256, JPEG q85)
!CLEAN_PATH=/kaggle/working/dataset_clean_resnet50 python repo/code/v1/data/02_clean_dataset.py

Étape 1/3 : identification des doublons...
Doublons à exclure : 3 | Fuites entre splits : 3

Étape 2/3 : resize à 256x256 + compression JPEG q=85...
Images traitées : 21010 | Converties RGB : 626

Étape 3/3 : rapport final...
train: {'corrosion': 3936, 'spalling': 3747, 'normal': 3594, 'cracks': 3430}
val: {'corrosion': 844, 'spalling': 802, 'normal': 770, 'cracks': 735}
test: {'corrosion': 844, 'spalling': 803, 'normal': 771, 'cracks': 734}

✅ Dataset nettoyé : /kaggle/working/dataset_clean_resnet50


In [4]:
!ls /kaggle/input
!ls /kaggle/input/datasets 2>/dev/null
# Garde-fou : arrête l'exécution si le nettoyage a échoué (évite des heures d'entraînement inutiles)
import os
root='/kaggle/working/dataset_clean_resnet50'
n={sp:sum(len(os.listdir(os.path.join(root,sp,c))) for c in os.listdir(os.path.join(root,sp))) for sp in ('train','val','test')}
print(n)
assert n=={'train': 14707, 'val': 3151, 'test': 3152}, 'effectifs inattendus : ' + str(n) + ' (attendu 14707 / 3151 / 3152)'

datasets
daikhiyassine
{'train': 14707, 'val': 3151, 'test': 3152}


In [5]:
# 2. Manifeste de partition (attendu : train=14707 val=3151 test=3152)
!python repo/tools/make_manifest.py /kaggle/working/dataset_clean_resnet50 /kaggle/working/v1_clean_resnet50_manifest.csv

21010 images -> /kaggle/working/v1_clean_resnet50_manifest.csv | train=14707 val=3151 test=3152
par classe/partition: {('train', 'corrosion'): 3936, ('train', 'cracks'): 3430, ('train', 'normal'): 3594, ('train', 'spalling'): 3747, ('val', 'corrosion'): 844, ('val', 'cracks'): 735, ('val', 'normal'): 770, ('val', 'spalling'): 802, ('test', 'corrosion'): 844, ('test', 'cracks'): 734, ('test', 'normal'): 771, ('test', 'spalling'): 803}


In [6]:
# 3. Hyperparamètres d'origine (pas de nouvelle recherche)
!cp repo/models/aigis_resnet50/v1/Hyperparametres/best_hyperparameters_resnet50.json /kaggle/working/best_hyperparameters_resnet50.json
!cat /kaggle/working/best_hyperparameters_resnet50.json

{
  "dense_units": 1024,
  "dropout_rate": 0.6000000000000001,
  "learning_rate": 0.001,
  "unfreeze_layers": 20,
  "use_batchnorm": true,
  "optimizer": "rmsprop"
}

In [7]:
# 4. Entraînement
!python repo/code/v1/resnet50/train.py

Hyperparamètres utilisés : {'dense_units': 1024, 'dropout_rate': 0.6000000000000001, 'learning_rate': 0.001, 'unfreeze_layers': 20, 'use_batchnorm': True, 'optimizer': 'rmsprop'}
Found 14707 files belonging to 4 classes.
Found 3151 files belonging to 4 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model: "functional_1"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼──────

In [8]:
# 5. Évaluation sur le split test
!python repo/code/v1/resnet50/test.py

Found 3152 files belonging to 4 classes.
              precision    recall  f1-score   support

   corrosion     0.9703    0.9301    0.9498       844
      cracks     0.9150    0.9087    0.9118       734
      normal     1.0000    0.9987    0.9994       771
    spalling     0.8637    0.9078    0.8852       803

    accuracy                         0.9362      3152
   macro avg     0.9373    0.9363    0.9366      3152
weighted avg     0.9375    0.9362    0.9366      3152

Test Accuracy : 0.9362 | AUC macro : 0.9915
Figure(800x600)
✅ Résultats sauvegardés


In [9]:
# 6. Environnement figé + empreintes des fichiers produits
!python repo/tools/capture_env.py kaggle_v1_resnet50_rerun
!python repo/tools/hash_files.py /kaggle/working/aigis_resnet50_final.keras
!cat environments/kaggle_v1_resnet50_rerun_env.json

{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "nvidia_smi": "Tesla T4, 580.178.04, 15360 MiB\nTesla T4, 580.178.04, 15360 MiB",
  "nvcc": [
    "Build cuda_13.0.r13.0/compiler.36424714_0"
  ],
  "tensorflow": "2.21.0",
  "keras": "3.13.2",
  "keras_tuner": "1.4.8",
  "torch": "2.11.0+cu130",
  "torch_cuda": "13.0",
  "torchvision": "0.26.0+cu130",
  "ultralytics": null,
  "sklearn": "1.6.1",
  "numpy": "2.1.3",
  "pandas": "2.3.3",
  "PIL": "13.0.0"
}
| `aigis_resnet50_final.keras` | 147591208 | `dc5bed70ed5056f8dfe97101292b8dd795a903c380fc8476615ae272a66633db` |
{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "nvidia_smi": "Tesla T4, 580.178.04, 15360 MiB\nTesla T4, 580.178.04, 15360 MiB",
  "nvcc": [
    "Build cuda_13.0.r13.0/compiler.36424714_0"
  ],
  "tensorflow": "2.21.0",
  "keras": "3.13.2",
  "keras_tuner": "1.4.8",
  "torch

In [7]:
!ls /kaggle/input/datasets/daikhiyassine/dataset

corrosion  cracks  normal  spalling  test  train  val


In [8]:
!rm -rf /kaggle/working/dataset_clean_resnet50